In [85]:
import pandas as pd
import json

# Caminho para seu arquivo
file_path = 'similarity_results.jsonl'

# Carrega cada linha do arquivo como um dicionário
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        # Extrai campos desejados
        row = {
            'similarity_result': item['similarity_result'],
            'diagnosed_name': item['diagnosed']['name'],
            'diagnosed_given': item['diagnosed']['given'],
            'diagnosed_then': item['diagnosed']['then'],
            'candidate_name': item['candidate']['name'],
            'candidate_given': item['candidate']['given'],
            'candidate_then': item['candidate']['then'],
            'elapsed_time': item['elapsed_time'],
            'config': item['config'],
        }
        data.append(row)

# Cria o DataFrame
df = pd.DataFrame(data)

# Exemplo: mostra as primeiras linhas
df


,similarity_result,diagnosed_name,diagnosed_given,diagnosed_then,candidate_name,candidate_given,candidate_then,elapsed_time,config
0,0.914141,scen_G1_T1,h < 100,delta_dt >= 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.056338,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
1,0.858585,scen_G1_T2,h < 100,h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.062928,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
2,0.916666,scen_G1_T3,h < 100,delta_dt >= 10 AND h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.066373,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
3,0.777778,scen_G1_T5,h < 100,delta_dt >= 10 AND p6 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.058172,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
4,0.722221,scen_G1_T6,h < 100,h >= 100 AND p6 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.055537,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
...,...,...,...,...,...,...,...,...,...
571,0.488275,scen_G31_T26,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,h >= 100 AND p7 > 10 AND p8 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.060885,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
572,0.576549,scen_G31_T27,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,delta_dt >= 10 AND h >= 100 AND p7 > 10 AND p8...,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.066726,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
573,0.516608,scen_G31_T29,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,delta_dt >= 10 AND p6 > 10 AND p7 > 10 AND p8 ...,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.054276,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."
574,0.461052,scen_G31_T30,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,h >= 100 AND p6 > 10 AND p7 > 10 AND p8 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.060316,"{'parameter_weights': {'parameter_1': 1.0, 'pa..."


In [86]:
# ---- 1. Definir baseline e função de parsing ----

baseline_given = "h < 100 AND vibration >= 63"
baseline_then  = "delta_dt >= 10 AND h >= 100"

def parse_clause(clause: str):
    # Quebra pela string "AND" e remove espaços extras
    return {part.strip() for part in clause.split("AND") if part.strip()}

# Conjunto estrutural do baseline
S_B = parse_clause(baseline_given) | parse_clause(baseline_then)
baseline_size = len(S_B)   # deve ser 4

# ---- 1.1. Descobrir o universo de átomos a partir do próprio df ----
all_atoms = set()
for _, row in df.iterrows():
    all_atoms |= parse_clause(row["diagnosed_given"])
    all_atoms |= parse_clause(row["diagnosed_then"])

# Átomos que podem ser "extras" (estão no universo, mas não no baseline)
extra_atoms = all_atoms - S_B
max_excess_size = len(extra_atoms)  # denominador p/ normalizar o excess

print("Baseline atoms:", S_B)
print("Extra atoms:", extra_atoms)
print("baseline_size:", baseline_size, "max_excess_size:", max_excess_size)

def structural_overlap_excess(given_str: str, then_str: str):
    # Conjunto estrutural do diagnosticado
    S_D = parse_clause(given_str) | parse_clause(then_str)

    shared = S_B & S_D      # interseção (cláusulas preservadas)
    excess = S_D - S_B      # cláusulas extras que não existem no baseline

    overlap = len(shared) / baseline_size

    if max_excess_size > 0:
        excess_norm = len(excess) / max_excess_size
    else:
        excess_norm = 0.0   # não há extras possíveis, então excess é sempre 0

    return overlap, excess_norm

# ---- 2. Aplicar no dataframe linha a linha ----

df["struct_overlap"], df["struct_excess"] = zip(
    *df.apply(
        lambda row: structural_overlap_excess(
            row["diagnosed_given"],
            row["diagnosed_then"]
        ),
        axis=1
    )
)

df


Baseline atoms: {'h < 100', 'vibration >= 63', 'delta_dt >= 10', 'h >= 100'}
Extra atoms: {'p3 > 10', 'p6 > 10', 'p8 > 10', 'p7 > 10', 'p1 > 10', 'p2 > 10'}
baseline_size: 4 max_excess_size: 6


,similarity_result,diagnosed_name,diagnosed_given,diagnosed_then,candidate_name,candidate_given,candidate_then,elapsed_time,config,struct_overlap,struct_excess
0,0.914141,scen_G1_T1,h < 100,delta_dt >= 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.056338,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.50,0.000000
1,0.858585,scen_G1_T2,h < 100,h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.062928,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.50,0.000000
2,0.916666,scen_G1_T3,h < 100,delta_dt >= 10 AND h >= 100,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.066373,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.75,0.000000
3,0.777778,scen_G1_T5,h < 100,delta_dt >= 10 AND p6 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.058172,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.50,0.166667
4,0.722221,scen_G1_T6,h < 100,h >= 100 AND p6 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.055537,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.50,0.166667
...,...,...,...,...,...,...,...,...,...,...,...
571,0.488275,scen_G31_T26,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,h >= 100 AND p7 > 10 AND p8 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.060885,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.75,0.833333
572,0.576549,scen_G31_T27,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,delta_dt >= 10 AND h >= 100 AND p7 > 10 AND p8...,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.066726,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",1.00,0.833333
573,0.516608,scen_G31_T29,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,delta_dt >= 10 AND p6 > 10 AND p7 > 10 AND p8 ...,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.054276,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.75,1.000000
574,0.461052,scen_G31_T30,h < 100 AND vibration >= 63 AND p1 > 10 AND p2...,h >= 100 AND p6 > 10 AND p7 > 10 AND p8 > 10,cand_low_speed_flying,h < 100 AND vibration >= 50 AND wind_speed > 60,delta_dt >= 10 AND h >= 90,0.060316,"{'parameter_weights': {'parameter_1': 1.0, 'pa...",0.75,1.000000


In [87]:
# ==============================
# 4) Scatter 3D: overlap x excess x similarity
# ==============================
import pandas as pd
import plotly.express as px

fig = px.scatter_3d(
    df,
    x="struct_overlap",
    y="struct_excess",
    z="similarity_result",
    color="similarity_result",               # colore pela similaridade
    hover_data=["diagnosed_name", "candidate_name"],  # info extra no hover
    title="3D Scatter: Similarity vs Structural Overlap / Structural Excess"
)

fig.update_layout(
    scene=dict(
        xaxis_title="Structural Overlap",
        yaxis_title="Structural Excess",
        zaxis_title="Similarity"
    )
)

fig.show()

In [102]:
import pandas as pd
import plotly.express as px
# ==============================
# 5) Heatmap "cara de artigo"
# ==============================

# ==============================
# 4) Construir o pivot (overlap x excess -> similarity média)
# ==============================

pivot = (
    df.pivot_table(
        index="struct_excess",
        columns="struct_overlap",
        values="similarity_result",
        aggfunc="mean"
    )
    .sort_index(axis=0)   # excess
    .sort_index(axis=1)   # overlap
)

print("pivot shape:", pivot.shape)

fig = px.imshow(
    pivot,
    labels=dict(
        x="Structural overlap (SO)",
        y="Structural excess (SE)",
        color="Mean similarity"
    ),
    aspect="auto",
    origin="lower",
    text_auto=".2f"  # pode remover se ficar poluído
)

fig.update_layout(
    title=None,
    width=600,
    height=500,
    font=dict(
        family="serif",
        size=14
    ),
    margin=dict(l=80, r=20, t=20, b=60),
    xaxis=dict(
        showgrid=False,
        ticks="outside",
        ticklen=5,
        tickfont=dict(size=12),
        autorange="reversed"   # X: 1 → 0
    ),
    yaxis=dict(
        showgrid=False,
        ticks="outside",
        ticklen=5,
        tickfont=dict(size=12),
        autorange="reversed"   # Y: 1 → 0
    ),
    coloraxis_colorbar=dict(
        title="Similarity",
        ticks="outside",
        ticklen=5,
        tickfont=dict(size=12)
    )
)

fig.update_xaxes(
    tickmode="array",
    tickvals=sorted(pivot.columns.unique()),
)
fig.update_yaxes(
    tickmode="array",
    tickvals=sorted(pivot.index.unique()),
)

fig.update_yaxes(
    tickmode="array",
    tickvals=sorted(pivot.index.unique()),
    tickformat=".2f",   # <-- só 2 casas decimais no eixo Y
)

# deixa tudo com fonte maior por padrão
fig.update_layout(
    font=dict(
        # family="Arial Black, Arial",  # Arial Black é bem mais "grossa"
        size=18,                      # aumenta o tamanho base
    )
)

# eixo X: título e ticks mais fortes
fig.update_xaxes(
    title_font=dict(size=18),  # título do eixo X
    tickfont=dict(size=14)     # números do eixo X
)

# eixo Y: título e ticks mais fortes
fig.update_yaxes(
    title_font=dict(size=18),  # título do eixo Y
    tickfont=dict(size=14)     # números do eixo Y
)



fig.show()

pivot shape: (7, 3)


In [62]:
import pandas as pd
import matplotlib.pyplot as plt

# (opcional) garantir que só ficam os 3 candidatos de interesse
cands_interesse = [
    "cand_hover_mode_flying",
    "cand_limited_satellite_flying",
    "cand_low_speed_flying",
]
df = df[df["candidate_name"].isin(cands_interesse)]

# 2) Definir ordem do eixo X (diagnósticos)
# pega todos os nomes únicos
diagnostics = df["diagnosed_name"].unique().tolist()

# coloca o baseline primeiro e o resto em ordem alfabética
diagnostics_order = ["diag_baseline"] + sorted(
    [d for d in diagnostics if d != "diag_baseline"]
)

# 3) Plotar com plotly.express
fig = px.line(
    df,
    x="diagnosed_name",
    y="similarity_result",
    color="candidate_name",
    markers=True,
    category_orders={"diagnosed_name": diagnostics_order},
)

fig.update_layout(
    title="Slope Chart - Mudança de Similaridade dos Candidatos Selecionados",
    xaxis_title="Execução Diagnóstica",
    yaxis_title="Similaridade",
)

fig.update_xaxes(tickangle=-45)

fig.show()

In [45]:
df_baseline = df[df['diagnosed_name'] == 'diag_baseline']
df_baseline

import plotly.express as px

fig = px.scatter(
    df_baseline,
    x='candidate_name',           # eixo X: nome do candidato
    y='similarity_result',        # eixo Y: similaridade
    color='similarity_result',    # cor dos pontos pela similaridade
    hover_data=['candidate_given', 'candidate_then'],  # informações extras no hover
    title='Similaridade por Candidato (Diagnosed Baseline)'
)
fig.update_layout(
    xaxis_title='Candidato',
    yaxis_title='Similaridade',
    template='plotly_white'
)
fig.show()

In [46]:
import pandas as pd
import plotly.graph_objects as go

# 3. Ordem das execuções (ajuste conforme necessário)
diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

# 5. Montar tabela pivô
df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]

# 6. Plotar o Slope Chart
fig = go.Figure()
for cand in df_pivot.index:
    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=df_pivot.loc[cand],
        mode='lines+markers',
        name=str(cand)
    ))
fig.update_layout(
    title='Slope Chart - Mudança de Similaridade dos Candidatos Selecionados',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Similaridade',
    xaxis=dict(tickangle=-45)
)
fig.show()


In [47]:
import pandas as pd
import plotly.graph_objects as go

# Carregar e preparar os dados
diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]

# Gerar o Slope Chart com melhor proporção e destaque visual
fig = go.Figure()
for cand in df_pivot.index:
    fig.add_trace(go.Scatter(
        x=df_pivot.columns,
        y=df_pivot.loc[cand],
        mode='lines+markers+text',
        text=[f'{y:.2f}' for y in df_pivot.loc[cand]],  # Mostra o valor em cada ponto
        textposition="top center",
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=12)
    ))
fig.update_layout(
    title='Slope Chart: Mudança de Similaridade dos Candidatos Selecionados',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Similaridade',
    xaxis=dict(tickangle=-45),
    width=850,   # largura reduzida
    height=600,  # altura aumentada
    legend=dict(
        title="Candidatos",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    margin=dict(l=40, r=40, t=70, b=90)
)
fig.show()


In [48]:
import pandas as pd
import plotly.graph_objects as go

# Carregar e preparar os dados
diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

# Montar matriz pivô
df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]

# Gerar ranking (1 = topo) para cada execução
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

# Plotar o bump chart
fig = go.Figure()
for cand in df_rank.index:
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=["" if i not in [0, len(df_rank.columns)-1] else str(cand) for i in range(len(df_rank.columns))],
        textposition="bottom center"
    ))
fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),  # 1 no topo!
    xaxis=dict(tickangle=-45),
    width=900,
    height=650,
    legend=dict(title="Candidatos", yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=40, r=40, t=70, b=90)
)
fig.show()


In [60]:
import pandas as pd
import plotly.express as px

# 2) Calcular rank por diagnosticado (1 = maior similaridade)
df["rank"] = (
    df.groupby("diagnosed_name")["similarity_result"]
      .rank(ascending=False, method="first")  # ou "min", se preferir
)

# 3) (Opcional) definir ordem do eixo X (diagnosticados)
# Se você quiser uma ordem específica, defina aqui:
diag_order = [
    "then_-delta_dt",
    "given_-h",
    "diag_baseline",
    "given_+windspeed",
    "given_+windspeed+p1",
    "given_+windspeed+p1+p2",
    "then_+satellite_count",
    "then_+satellite_count+delta_flight_time",
    "then_+satellite_count+delta_flight_time+p3",
    "then_+satellite_count+delta_flight_time+p3+p4",
]
# Se algum nome da lista não existir no CSV, comente essa parte ou ajuste a lista.
diag_order = [d for d in diag_order if d in df["diagnosed_name"].unique()]

# 4) Criar o slope chart (rank vs diagnosticado)
fig = px.line(
    df,
    x="diagnosed_name",
    y="rank",
    color="candidate_name",
    markers=True,
    text="similarity_result",   # texto em cada ponto = similaridade
    category_orders={"diagnosed_name": diag_order} if diag_order else None,
)

# 5) Ajustes de layout
fig.update_yaxes(
    autorange="reversed",  # rank 1 fica em cima (como no seu gráfico)
    title_text="Rank"
)
fig.update_xaxes(title_text="Diagnostic Scenario")

# Formatar o texto (similaridade) em duas casas decimais, acima do ponto
fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="top center"
)

fig.update_layout(
    title="Variação do Rank dos Candidatos por Diagnóstico",
    legend_title_text="Candidate",
)

fig.show()


In [49]:
import pandas as pd
import plotly.graph_objects as go

diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()
for cand in df_rank.index:
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=["" if i not in [0, len(df_rank.columns)-1] else str(cand) for i in range(len(df_rank.columns))],
        textposition="bottom center"
    ))
fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=diagnosis_order,
        tickvals=diagnosis_order,
        ticktext=diagnosis_order
    ),
    width=1500,   # largura aumentada!
    height=700,
    legend=dict(title="Candidatos", yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=40, r=40, t=70, b=90)
)
fig.show()


In [50]:
import pandas as pd
import plotly.graph_objects as go

# Carregar e preparar os dados

diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()
for cand in df_rank.index:
    similarity_texts = [f"{df_pivot.loc[cand, col]:.2f}" for col in df_pivot.columns]
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=10)  # <<< Ajuste aqui: tamanho do texto!
    ))
fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=diagnosis_order,
        tickvals=diagnosis_order,
        ticktext=diagnosis_order
    ),
    width=1500,
    height=700,
    legend=dict(title="Candidatos", yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=40, r=40, t=70, b=90)
)
fig.show()


In [51]:
import pandas as pd
import plotly.graph_objects as go

diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()
for cand in df_rank.index:
    similarity_texts = [f"{df_pivot.loc[cand, col]:.2f}" for col in df_pivot.columns]
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=10)
    ))

# Indica a posição central do diagnosed_baseline
baseline_idx = diagnosis_order.index("diag_baseline")

# Adiciona duas linhas tracejadas finas (antes e depois do baseline)
fig.add_vline(
    x=baseline_idx - 0.48,  # um pouco antes
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)
fig.add_vline(
    x=baseline_idx + 0.48,  # um pouco depois
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)

fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=diagnosis_order,
        tickvals=diagnosis_order,
        ticktext=diagnosis_order
    ),
    width=1500,
    height=700,
    legend=dict(title="Candidatos", yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=40, r=40, t=70, b=90)
)
fig.show()


In [52]:
import pandas as pd
import plotly.graph_objects as go

# Carregar e preparar os dados
diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

# Supondo que seu DataFrame df já está filtrado para os candidatos desejados!
df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()

for cand in df_rank.index:
    similarity_texts = [f"{df_pivot.loc[cand, col]:.2f}" for col in df_pivot.columns]
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=10)
    ))

# Linhas verticais tracejadas finas ao redor do diagnosed_baseline
baseline_idx = diagnosis_order.index("diag_baseline")
fig.add_vline(
    x=baseline_idx - 0.48,
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)
fig.add_vline(
    x=baseline_idx + 0.48,
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)

fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=diagnosis_order,
        tickvals=diagnosis_order,
        ticktext=diagnosis_order
    ),
    width=1500,
    height=700,
    legend=dict(
        title="Candidatos",
        yanchor="top", y=1,
        xanchor="left", x=1.01,      # Legenda à direita, fora do gráfico
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=13)
    ),
    margin=dict(l=40, r=140, t=70, b=90)  # margem direita maior
)
fig.show()


In [53]:
import pandas as pd
import plotly.graph_objects as go

diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then", 
    "remove_90_given", "remove_60_given", "remove_30_given", 
    "diag_baseline", 
    "add_30_given", "add_60_given", "add_90_given", 
    "add_30_then", "add_60_then", "add_90_then"
]

# Supondo que seu DataFrame df já está filtrado para os candidatos desejados!
df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")
df_pivot = df_pivot[[col for col in diagnosis_order if col in df_pivot.columns]]
df_rank = df_pivot.rank(ascending=False, axis=0, method='min')

fig = go.Figure()

for cand in df_rank.index:
    similarity_texts = [f"{df_pivot.loc[cand, col]:.2f}" for col in df_pivot.columns]
    fig.add_trace(go.Scatter(
        x=df_rank.columns,
        y=df_rank.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=10)
    ))

# Linhas verticais tracejadas finas ao redor do diag_baseline
baseline_idx = diagnosis_order.index("diag_baseline")
fig.add_vline(
    x=baseline_idx - 0.48,
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)
fig.add_vline(
    x=baseline_idx + 0.48,
    line_width=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5,
    layer='above'
)

fig.update_layout(
    title='Bump Chart - Ranking dos Candidatos ao Longo das Execuções',
    xaxis_title='Execução Diagnóstica',
    yaxis_title='Ranking (1 = topo)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=diagnosis_order,
        tickvals=diagnosis_order,
        ticktext=diagnosis_order
    ),
    width=1000,    # <<< Espaçamento reduzido entre X!
    height=700,
    legend=dict(
        title="Candidatos",
        yanchor="top", y=1,
        xanchor="left", x=1.01,
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=13)
    ),
    margin=dict(l=40, r=140, t=70, b=90)
)
fig.show()


In [54]:
import pandas as pd
import plotly.graph_objects as go

# Ordem original completa (mantida só para referência)
diagnosis_order = [
    "remove_90_then", "remove_60_then", "remove_30_then",
    "remove_90_given", "remove_60_given", "remove_30_given",
    "diag_baseline",
    "add_30_given", "add_60_given", "add_90_given",
    "add_30_then", "add_60_then", "add_90_then"
]

# 1) Ordem somente com 'given' + baseline (removals → baseline → additions)
given_only_order = [
    "remove_90_given", "remove_60_given", "remove_30_given",
    "diag_baseline",
    "add_30_given", "add_60_given", "add_90_given"
]

# df deve ter colunas: candidate_name, diagnosed_name, similarity_result
df_pivot = df.pivot(index="candidate_name", columns="diagnosed_name", values="similarity_result")

# 2) Seleciona apenas as colunas 'given' + baseline na ordem desejada
given_only_order = [col for col in given_only_order if col in df_pivot.columns]
df_pivot_given = df_pivot[given_only_order]

# 3) Recalcula ranking só para essas execuções
df_rank_given = df_pivot_given.rank(ascending=False, axis=0, method='min')

fig = go.Figure()

for cand in df_rank_given.index:
    similarity_texts = [f"{df_pivot_given.loc[cand, col]:.2f}" for col in df_pivot_given.columns]
    fig.add_trace(go.Scatter(
        x=df_rank_given.columns,
        y=df_rank_given.loc[cand],
        mode='lines+markers+text',
        name=str(cand),
        line=dict(width=4),
        marker=dict(size=13),
        text=similarity_texts,
        textposition="top center",
        textfont=dict(size=10)
    ))

# 4) Linhas verticais tracejadas ao redor do baseline (recalculadas para o novo eixo X)
if "diag_baseline" in given_only_order:
    baseline_idx = given_only_order.index("diag_baseline")
    fig.add_vline(x=baseline_idx - 0.48, line_width=1, line_dash="dash",
                  line_color="black", opacity=0.5, layer='above')
    fig.add_vline(x=baseline_idx + 0.48, line_width=1, line_dash="dash",
                  line_color="black", opacity=0.5, layer='above')

# 5) Layout (legenda horizontal abaixo do eixo X)
fig.update_layout(
    title='Bump Chart — Ranking (only GIVEN perturbations)',
    # xaxis_title='Diagnostic execution',
    yaxis_title='Rank (1 = top)',
    yaxis=dict(autorange='reversed', dtick=1),
    xaxis=dict(
        tickangle=-45,
        categoryorder='array',
        categoryarray=given_only_order,
        tickvals=given_only_order,
        ticktext=given_only_order
    ),
    width=1000,
    height=700,
    legend=dict(
        title="Candidates",
        orientation="h",            # horizontal
        xanchor="center", x=0.5,    # centraliza
        yanchor="top",    y=-0.20,  # posiciona abaixo do eixo X
        bgcolor="rgba(255,255,255,0.95)",
        bordercolor="#ccc",
        borderwidth=1,
        font=dict(size=13)
    ),
    margin=dict(l=40, r=40, t=70, b=160)  # margem inferior maior p/ legenda
)

fig.show()
